# TUE Reimbursement

This notebook demonstrates goal-oriented alignment visualization for the TUE Reimbursment case study.

- Models are loaded in the notebook.
- Handmade test cases are defined inline based on the traces discussed in the paper.



## Setup & Imports

In [1]:
import pandas as pd

import pm4py
from pm4py.objects.conversion.log import converter as log_converter
from pm4py.objects.log.obj import EventLog


from Semantics.goccva_pipeline import analyse

from Ui.goccva_ui import render_from_analysis, render_all_goal_oriented_alignments_from_analysis, render_case_distribution_matrix

from Semantics.goccva_helpers import sequences_to_event_log

from Semantics.istar_processor import read_istar_model
from Semantics.petri_net_processor import read_petri_net
from Semantics.event_mapping_from_csv import read_event_mapping_csv

from Semantics.goccva_filter import ComplianceStatus, TraceFilter

from pprint import pp

## Load Models

Load the goal model, process model, and mapping used by the case study.

In [2]:

# Paths used in the GoCCvA repository
goal_model_path = "content/TUEReimbursement/GMjcavi2.txt"
process_model_path = "content/TUEReimbursement/domestic_declaration_ilpn_updated.pnml"
mapping_path = "content/TUEReimbursement/mappingjcavi.csv"

goal_model = read_istar_model(str(goal_model_path), qualified=True)

petri_net = read_petri_net(str(process_model_path))

activity_mapping = read_event_mapping_csv(str(mapping_path))

# Fix minor inconsistencies in the mapping vs. the goal model (e.g., extra spaces, missing/extra parentheses, etc.)
# This should not be required as I have changed GMjcavi2.txt to make it compatiible with the mapping.
activity_mapping = goal_model.canonicalize_activity_mapping(activity_mapping) 


## Target Configuration

Define the target requirements to be evaluated. The make, break, and non-related sets are computed from the loaded goal model.

In [3]:
targets = [
    '(Admin) adequate declaration handling',
    '(Employee) Increase employee satisfaction',
]

## Activity Abbreviations

In [4]:
activity_abbreviations = {
    "Declaration REJECTED by ADMINISTRATION": "ra",
    "Payment Handled": "ph",
    "Declaration REJECTED by BUDGET OWNER": "rb",
    "Declaration SAVED by EMPLOYEE": "dsv",
    "Declaration APPROVED by ADMINISTRATION": "aa",
    "Declaration REJECTED by EMPLOYEE": "er",
    "Request Payment": "rp",
    "Declaration SUBMITTED by EMPLOYEE": "ds",
    "Declaration APPROVED by BUDGET OWNER": "ba",
    "Declaration FINAL_APPROVED by SUPERVISOR": "as",
    "Declaration REJECTED by SUPERVISOR": "rs",
    "Declaration APPROVED by PRE_APPROVER": "pa",
    "Declaration REJECTED by PRE_APPROVER": "rpa",
    "Declaration REJECTED by MISSING": "rm",
    "t_tau_rev": "tau",
}

print("Activity abbreviations configured")


Activity abbreviations configured


# Reading the .xes file


In [5]:
log_file_path = "content/TUEReimbursement/DomesticDeclarations.xes.gz"

full_log = log_converter.apply(pm4py.read_xes(str(log_file_path)), variant=log_converter.Variants.TO_EVENT_LOG)

print("Log file loaded")
print(len(full_log))

/Users/huba/miniconda3/envs/kogi/lib/python3.11/site-packages/pm4py/utils.py:987: UserWarning: In the current version, the import/export operation uses `rustxes` by default for importing/exporting files faster. Please uninstall `rustxes` to revert the behavior.
  warnings.warn("In the current version, the import/export operation uses `rustxes` by default for importing/exporting files faster. Please uninstall `rustxes` to revert the behavior.")


Log file loaded
10357


In [6]:
summary, detailed, contribution_to_targets = analyse(
    goal_model,
    petri_net,
    full_log,
    targets,
    activity_mapping,
    initial_marking=None,
 )

matrix_result = render_case_distribution_matrix(
    summary=summary,
    title="TUE Reimbursement Case Distribution Matrix",
    targets=targets,
)

print(matrix_result["counts"])

aligning log, completed variants ::   0%|          | 0/90 [00:00<?, ?it/s]

{'O+': 2411, 'O~': 0, 'O-': 185, 'N+': 1, 'N~': 314, 'N-': 7446}


## Compute the traces as a list of list of actions from the log.

In [7]:

traces = [[event['concept:name'] for event in trace] for trace in full_log]
print(f"{len(traces)} traces loaded from the log")
pp(traces[:1])


10357 traces loaded from the log
[['Declaration SUBMITTED by EMPLOYEE',
  'Declaration FINAL_APPROVED by SUPERVISOR',
  'Request Payment',
  'Payment Handled']]


## Apply a filter to the list traces

Get all the traces, where "(Employee) Increase employee satisfaction" is either strongly or weakly compliant.

In [8]:
trace_filter = TraceFilter(goal_model, traces, activity_mapping)

employee_satisfied_traces = (
    trace_filter
    .query()
    .where("(Employee) Increase employee satisfaction", ComplianceStatus.COMPLIANT)
    .traces()
)

print(f"{len(employee_satisfied_traces)} satisfy (strongly or weakly) the target '(Employee) Increase employee satisfaction'") 
print(f"This is {len(employee_satisfied_traces)*100/len(traces):.2f}% of all traces")
print(f"Only {(len(traces) - len(employee_satisfied_traces))*100/len(traces):.2f}% of traces do not satisfy the target '(Employee) Increase employee satisfaction'")

9911 satisfy (strongly or weakly) the target '(Employee) Increase employee satisfaction'
This is 95.69% of all traces
Only 4.31% of traces do not satisfy the target '(Employee) Increase employee satisfaction'


In [13]:
adequate_declaration_traces = (
    trace_filter
    .query()
    .where("(Employee) Increase employee satisfaction", ComplianceStatus.COMPLIANT)
    .where("(Admin) adequate declaration handling", ComplianceStatus.COMPLIANT)
    .traces()
)
print(f"However, only {len(adequate_declaration_traces)*100/len(employee_satisfied_traces):.2f}% of those traces satisfy the target '(Admin) adequate declaration handling'")
print(f"These are {len(adequate_declaration_traces)} traces")
print(f"Or {(len(traces)- len(adequate_declaration_traces))*100/len(traces):.2f}% of all traces do not satisfy both targets")

However, only 27.50% of those traces satisfy the target '(Admin) adequate declaration handling'
These are 2726 traces
Or 73.68% of all traces do not satisfy both targets


The following are the unique traces with satisfy "(Employee) Increase employee satisfaction" and don't satisfy "(Admin) adequate declaration handling" and which contain the word BUDGET OWNER.

One can clearly see, that the budget owner only rejects declarations, but never approves them in case the "(Admin) adequate declaration handling" is not satisfied.

In [10]:
budget_owner_traces = (
    trace_filter
    .query()
    .where("(Employee) Increase employee satisfaction", ComplianceStatus.COMPLIANT)
    .where("(Admin) adequate declaration handling", ComplianceStatus.NON_COMPLIANT)
    .contains("BUDGET OWNER")
    .unique()
    .traces()
)

pp(len(budget_owner_traces))
pp(budget_owner_traces)

3
[['Declaration SUBMITTED by EMPLOYEE',
  'Declaration APPROVED by ADMINISTRATION',
  'Declaration REJECTED by BUDGET OWNER',
  'Declaration REJECTED by EMPLOYEE',
  'Declaration SUBMITTED by EMPLOYEE',
  'Declaration APPROVED by ADMINISTRATION',
  'Declaration FINAL_APPROVED by SUPERVISOR',
  'Request Payment',
  'Payment Handled'],
 ['Declaration SUBMITTED by EMPLOYEE',
  'Declaration APPROVED by ADMINISTRATION',
  'Declaration REJECTED by SUPERVISOR',
  'Declaration REJECTED by EMPLOYEE',
  'Declaration SUBMITTED by EMPLOYEE',
  'Declaration APPROVED by ADMINISTRATION',
  'Declaration REJECTED by BUDGET OWNER',
  'Declaration REJECTED by EMPLOYEE',
  'Declaration SUBMITTED by EMPLOYEE',
  'Declaration APPROVED by ADMINISTRATION',
  'Declaration REJECTED by BUDGET OWNER',
  'Declaration REJECTED by EMPLOYEE',
  'Declaration SUBMITTED by EMPLOYEE',
  'Declaration APPROVED by ADMINISTRATION',
  'Declaration FINAL_APPROVED by SUPERVISOR',
  'Request Payment',
  'Payment Handled'],
 ['D

In contrast, 100% of the traces that satisfy both targets also contain an approval by the budget owner.

In [ ]:
budget_owner_approves_traces = (
    trace_filter
    .query()
    .where("(Employee) Increase employee satisfaction", ComplianceStatus.COMPLIANT)
    .where("(Admin) adequate declaration handling", ComplianceStatus.COMPLIANT)
    .contains("Declaration APPROVED by BUDGET OWNER")
    .traces()
)

print(f"This means that {len(budget_owner_approves_traces)*100/len(adequate_declaration_traces):.2f}% of the traces that satisfy both targets also contain an approval by the budget owner")
pp(len(budget_owner_approves_traces))

This means that 100.00% of the traces that satisfy both targets also contain an approval by the budget owner
2726
